# 01 — Análise das ações do Ibovespa

Esse notebook aborda:

> TODO (Gabriel): listar as etapas do checklist cobertas aqui.

##  Roteiro e fontes

Este projeto é de **análise de dados**, não de *machine learning*: não há modelo, nem conjunto de treino e teste. Do checklist do Géron valem as etapas **1** (objetivo), **2** (dados), **3** (exploração) e **7** (apresentação). As tarefas numeradas **T1 a T10** são as do enunciado da Asimov (*Projeto: Análise de dados de ações*), reescritas como perguntas.

| Sigla | Fonte | Onde ler | Verificação |
|---|---|---|---|
| **Géron** | *Hands-On Machine Learning*, apêndice "Checklist de um projeto de Machine Learning" (tradução Asimov) | `o-checklist-de-um-projeto-de-machine-learning.pdf`, no zip *Fundamentos de AI e Machine Learning* | lido |
| **B3** | Ibovespa B3: carteira teórica, "reavaliado a cada quatro meses", rebalanceamento "Quadrimestral (1ª segunda-feira de Janeiro, Maio e Setembro)" | https://www.b3.com.br/pt_br/market-data-e-indices/indices/indices-amplos/ibovespa-b3.htm | lido |
| **pandas** | `DataFrame.pct_change`: "Fractional change between the current and a prior element"; resultado em fração, não em % | https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.pct_change.html | lido (pandas 3.1.1) |
| **yfinance** | `history(..., auto_adjust=True, actions=True, ...)`: "Adjust all OHLC automatically?"; `get_dividends()` | https://ranaroussi.github.io/yfinance/reference/yfinance.price_history.html | lido; versão instalada 1.8.0, `auto_adjust=True` por padrão |
| **DY** | *Dividend yield* = dividendos anualizados por ação ÷ preço atual; "Published dividend yields are not fully standardised" | https://en.wikipedia.org/wiki/Dividend_yield | lido |
| **Sobrevivência** | Viés de sobrevivência em índices: "the use of a current index membership set rather than using the actual constituent changes over time" | https://en.wikipedia.org/wiki/Survivorship_bias (seção *Finance and economics*); artigo clássico: Brown, Goetzmann, Ibbotson & Ross, *Survivorship Bias in Performance Studies*, Review of Financial Studies 5(4), 1992 | lido (o artigo, só os metadados) |
| **Backtest** | Limitações: dados históricos, impacto da própria estratégia e "potential overfitting" | https://en.wikipedia.org/wiki/Backtesting | lido |

## Setup

In [73]:
from pathlib import Path
from IPython.display import display, Markdown
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf


SEED = 42
DATA_DIR = Path("../data")

## 1. Objetivo

*Perguntas** (Géron, etapa 1):
- Qual é o objetivo: **descrever** o comportamento das ações ou **avaliar estratégias** de investimento? O enunciado pede as duas coisas (T1–T6 descrevem; T7, T8 e T10 avaliam estratégias). (1.1)
- Que decisão alguém tomaria com essas respostas, e qual seria o critério de "estratégia boa" (retorno médio, probabilidade de alta, retorno acumulado contra uma referência)? (1.5, 1.6)
- Qual é a **referência** de comparação para cada estratégia (ex.: comprar todos os ativos e segurar)? (1.7)
- Que suposições você está fazendo (ex.: dá para comprar pelo preço de fechamento, sem custo de corretagem nem imposto)? (1.11)

## 2. Obter e carregar os dados



**Perguntas — composição do índice (arquivo da B3):**
- De que data é a carteira, e por quanto tempo ela vale? O Ibovespa é **rebalanceado a cada quatro meses** (B3). (Géron 2.10)
- O que significam `Part. (%)` e `Qtde. Teórica`? 
- Em que formato os números chegam: separador de milhar, vírgula decimal, texto? Qual parâmetro do `pd.read_csv` resolve na leitura em vez de corrigir depois (`decimal=`, `thousands=`)?
- Há espaços extras em colunas de texto (ex.: `Tipo`)? Isso importa para a análise?

In [74]:
df_acoes = pd.read_csv(DATA_DIR / "ibovdia_09-08-22.csv")
display(df_acoes.head(5))

,Código,Ação,Tipo,Qtde. Teórica,Part. (%)
0,RRRP3,3R PETROLEUM,ON NM,200.372.163,"0,341"
1,ALPA4,ALPARGATAS,PN N1,201.257.220,"0,193"
2,ABEV3,AMBEV S/A,ON,4.380.195.841,"3,214"
3,AMER3,AMERICANAS,ON NM,596.875.824,"0,421"
4,ASAI3,ASSAI,ON NM,794.531.367,"0,682"


In [75]:
display(pd.DataFrame({
    "Rows": [df_acoes.shape[0]],
    "Cols": [df_acoes.shape[1]]
    }))

display(df_acoes.info())
display(df_acoes.describe())
display(df_acoes.duplicated().sum())
nulos = df_acoes.isnull()
display(pd.DataFrame({
    "Valores nulos: ": nulos.sum(),
    "Porcentagem de nulos: ": (nulos.sum()/len(df_acoes))*100
}))


,Rows,Cols
0,90,5


<class 'pandas.DataFrame'>
RangeIndex: 90 entries, 0 to 89
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   Código         90 non-null     str  
 1   Ação           90 non-null     str  
 2   Tipo           90 non-null     str  
 3   Qtde. Teórica  90 non-null     str  
 4   Part. (%)      90 non-null     str  
dtypes: str(5)
memory usage: 3.6 KB


None

,Código,Ação,Tipo,Qtde. Teórica,Part. (%)
count,90,90,90,90,90
unique,90,87,17,90,87
top,RRRP3,BRADESCO,ON NM,200.372.163,"0,231"
freq,1,2,58,1,3


np.int64(0)

,Valores nulos:,Porcentagem de nulos:
Código,0,0.0
Ação,0,0.0
Tipo,0,0.0
Qtde. Teórica,0,0.0
Part. (%),0,0.0


## 3. Análise exploratória dos dados

In [76]:
df_acoes['Qtde. Teórica'] = pd.to_numeric(df_acoes["Qtde. Teórica"].str.replace('.', '', regex=False))
df_acoes["Part. (%)"] = pd.to_numeric(df_acoes["Part. (%)"].str.replace(',','.',regex=False))

In [77]:
display(df_acoes.dtypes)
display(df_acoes.drop(columns=["Qtde. Teórica", "Part. (%)"]).describe())
display(df_acoes.describe())

Código               str
Ação                 str
Tipo                 str
Qtde. Teórica      int64
Part. (%)        float64
dtype: object

,Código,Ação,Tipo
count,90,90,90
unique,90,87,17
top,RRRP3,BRADESCO,ON NM
freq,1,2,58


,Qtde. Teórica,Part. (%)
count,9.000000e+01,90.000000
mean,1.076093e+09,1.111111
std,1.278690e+09,1.839732
min,7.805372e+07,0.031000
25%,2.949136e+08,0.228750
50%,6.849525e+08,0.504000
75%,1.164388e+09,1.162500
max,5.987625e+09,12.817000


3.1 As 10 maiores ações e o download dos preços (T1–T3)

**Perguntas:**
- Quais são as **10 ações com maior `Part. (%)`**? A ordenação funciona com a coluna no tipo atual? (T1)
- Como transformar o `Código` da B3 no *ticker* do Yahoo (ex.: `PETR4` → `PETR4.SA`)? (T1)
- Que período baixar, e por quê? O que muda entre `period="max"` e um intervalo com `start`/`end`? (T2)
- Com `auto_adjust=True` (padrão do `yfinance` instalado), o preço de fechamento já vem **ajustado**. Ajustado a quê? Como conferir com `get_dividends()` e com os desdobramentos? Isso muda alguma resposta das tarefas seguintes? (T2; yfinance)
- Ao juntar os fechamentos num único DataFrame (uma coluna por ativo), as datas coincidem? O que aparece nos dias em que um ativo não negociou? (T3)

> **Enunciado do professor (T1):** Vamos analisar o histórico das 10 ações com maior participacão na bolsa. Encontre estes ativos no DataFrame `df_ibov` e carregue-os em um objeto Tickers do módulo do Yahoo.

> **Enunciado do professor (T2):** Crie um dicionário que conterá como chaves os tickers de cada ativo (PETR4.SA, por exemplo) e como valores um DataFrame contendo os dados históricos de cada

> **Enunciado do professor (T3):** Combine todos os dados em um único DataFrame, utilizando o preço de fechamento de cada ativo.

In [78]:
acoes_ordenadas = df_acoes.sort_values("Part. (%)").tail(10)["Código"].values
" ".join(i + ".SA" for i in acoes_ordenadas)

'RENT3.SA ELET3.SA BBAS3.SA ABEV3.SA B3SA3.SA BBDC4.SA PETR3.SA ITUB4.SA PETR4.SA VALE3.SA'

In [89]:
acoes_ordenadas = df_acoes.sort_values("Part. (%)").tail(10)["Código"].values
tickers = yf.Tickers(" ".join(i + ".SA" for i in acoes_ordenadas))
display(tickers)
display(tickers.tickers)

yfinance.Tickers object <RENT3.SA,ELET3.SA,BBAS3.SA,ABEV3.SA,B3SA3.SA,BBDC4.SA,PETR3.SA,ITUB4.SA,PETR4.SA,VALE3.SA>

{'RENT3.SA': yfinance.Ticker object <RENT3.SA>,
 'ELET3.SA': yfinance.Ticker object <ELET3.SA>,
 'BBAS3.SA': yfinance.Ticker object <BBAS3.SA>,
 'ABEV3.SA': yfinance.Ticker object <ABEV3.SA>,
 'B3SA3.SA': yfinance.Ticker object <B3SA3.SA>,
 'BBDC4.SA': yfinance.Ticker object <BBDC4.SA>,
 'PETR3.SA': yfinance.Ticker object <PETR3.SA>,
 'ITUB4.SA': yfinance.Ticker object <ITUB4.SA>,
 'PETR4.SA': yfinance.Ticker object <PETR4.SA>,
 'VALE3.SA': yfinance.Ticker object <VALE3.SA>}

### 3.2 Histórico em comum (T4)

**Perguntas:**
- Qual é a **primeira data em que todos** os 10 ativos têm preço? Que método encontra isso sem olhar coluna por coluna (ex.: `dropna()` e o primeiro índice)?
- Qual ativo limita essa data, e por quê (ex.: abriu capital mais tarde)?
- Daqui em diante, você analisa **só o período em comum** ou cada ativo no seu período? Que diferença isso faz nas comparações?

> **Enunciado do professor (T4):** Qual a primeira data na qual todos os ativos estudados já estavam na bolsa?

In [91]:
tickers_dict = {}

for ticker in tickers.tickers.keys():
    print("Capturndo informçaões do ticker: ", ticker)
    tickers_dict[ticker] = tickers.tickers[ticker].history(period="max")

Capturndo informçaões do ticker:  RENT3.SA
Capturndo informçaões do ticker:  ELET3.SA


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: ELET3.SA"}}}
$ELET3.SA: possibly delisted; no timezone found


Capturndo informçaões do ticker:  BBAS3.SA
Capturndo informçaões do ticker:  ABEV3.SA
Capturndo informçaões do ticker:  B3SA3.SA
Capturndo informçaões do ticker:  BBDC4.SA
Capturndo informçaões do ticker:  PETR3.SA
Capturndo informçaões do ticker:  ITUB4.SA
Capturndo informçaões do ticker:  PETR4.SA
Capturndo informçaões do ticker:  VALE3.SA


Código das tarefas T3 e T4

In [106]:
df_fechamento = pd.DataFrame()

for ticker in tickers_dict.keys():
    df_fechamento[ticker] = tickers_dict[ticker]["Close"].round(4)

display(df_fechamento)

,RENT3.SA,ELET3.SA,BBAS3.SA,ABEV3.SA,B3SA3.SA,BBDC4.SA,PETR3.SA,ITUB4.SA,PETR4.SA,VALE3.SA
Date,,,,,,,,,,
2005-05-23 00:00:00-03:00,0.7267,NaN,1.5834,0.9828,NaN,NaN,3.1294,2.6526,2.1563,6.1946
2005-05-24 00:00:00-03:00,0.7261,NaN,1.5940,0.9813,NaN,NaN,3.1666,2.6463,2.1955,6.3422
2005-05-25 00:00:00-03:00,0.7267,NaN,1.5913,0.9904,NaN,NaN,3.1880,2.6526,2.2123,6.2786
2005-05-27 00:00:00-03:00,0.7267,NaN,1.6330,0.9904,NaN,NaN,3.2567,2.6715,2.2606,6.3189
2005-05-30 00:00:00-03:00,0.7267,NaN,1.6282,0.9904,NaN,NaN,3.2576,2.6936,2.2762,6.3422
...,...,...,...,...,...,...,...,...,...,...
2026-11-13 00:00:00-03:00,30.0700,NaN,15.1000,15.7900,14.89,14.52,51.8700,42.2700,45.3000,63.5000
2026-11-16 00:00:00-03:00,29.3100,NaN,14.9500,15.8700,14.81,14.23,50.1900,41.3000,44.0200,60.7000
2026-11-17 00:00:00-03:00,29.6600,NaN,14.6200,15.9300,14.65,14.41,50.2300,40.8900,43.9800,60.3400


In [112]:
primeira_data = df_fechamento.drop(columns="ELET3.SA").dropna().index[0]
display(primeira_data)
display(pd.DataFrame({"Primeira data com preço": df_fechamento.notna().idxmax()}).sort_values("Primeira data com preço"))

df_comum = df_fechamento.loc[primeira_data:]
display(df_comum.shape)
display(tickers_dict["ELET3.SA"])

Timestamp('2008-01-02 00:00:00-0200', tz='America/Sao_Paulo')

,Primeira data com preço
RENT3.SA,2005-05-23 00:00:00-03:00
ELET3.SA,2005-05-23 00:00:00-03:00
BBAS3.SA,2005-05-23 00:00:00-03:00
ABEV3.SA,2005-05-23 00:00:00-03:00
PETR3.SA,2005-05-23 00:00:00-03:00
ITUB4.SA,2005-05-23 00:00:00-03:00
PETR4.SA,2005-05-23 00:00:00-03:00
VALE3.SA,2005-05-23 00:00:00-03:00
B3SA3.SA,2007-10-26 00:00:00-02:00
BBDC4.SA,2008-01-02 00:00:00-02:00


(4692, 10)

,Open,High,Low,Close,Adj Close,Volume
Date,,,,,,


#### Insights

- Primeira data com os 10 ativos: **02/01/2008**. Quem limita é o **BBDC4**, cujo histórico no Yahoo começa nesse dia. Foi necessário retirar a coluna "ELET3.SA" que apresentava apenas valores nulos.


### 3.3 Variação diária (T5)

**Perguntas:**
- O que `pct_change()` calcula, e em que unidade? (pandas: fração, não porcentagem — multiplique por 100 para %.)
- Por que a primeira linha sai `NaN`? O que acontece com dias sem preço no meio da série? (No pandas 3.1, o `pct_change` **não preenche** valores ausentes.)
- Como é a distribuição das variações diárias de cada ativo: média, desvio, assimetria? Algum ativo é muito mais volátil? (Géron 3.3, 3.5)

> **Enunciado do professor (T5):** Crie um DataFrame adicional para representar a variação percentual diária de cada ativo

 Código da tarefa T5

In [83]:
df_variacao = df_comum.pct_change()

display(pd.DataFrame({
    "Média diária (%)": df_variacao.mean() * 100,
    "Desvio diário (%)": df_variacao.std() * 100,
    "Assimetria": df_variacao.skew(),
}).round(3))

,Média diária (%),Desvio diário (%),Assimetria
RENT3.SA,0.084,2.745,-0.238
AXIA3.SA,0.076,3.008,1.185
BBAS3.SA,0.057,2.579,-0.225
ABEV3.SA,0.056,1.702,0.142
B3SA3.SA,0.043,2.918,-2.680
BBDC4.SA,0.036,2.325,-1.285
PETR3.SA,0.070,2.812,-0.212
ITUB4.SA,0.065,2.128,0.423
PETR4.SA,0.073,2.759,-0.308
VALE3.SA,0.055,2.578,0.142


### 3.4 Piores oscilações (T6)

**Perguntas:**
- Em que **data** cada ativo teve a maior queda diária, e de quanto foi? Que método devolve a data do mínimo de cada coluna (`idxmin`)?
- As piores datas coincidem entre os ativos? Se coincidem, o que aconteceu no mercado nesse dia? (Compare com o noticiário da época.)
- Alguma queda é tão grande que parece **erro de dado** ou desdobramento mal ajustado? Como verificar? (Géron 3.3, ruído)

> **Enunciado do professor (T6):** Quais foram as datas na qual cada ativo apresentou sua pior oscilação? Apresente também a variação percentual.

Código da tarefa T6

In [113]:
display(pd.DataFrame({
    "Data da pior queda": df_variacao.idxmin(),
    "Variação (%)": (df_variacao.min() * 100).round(2),
}).sort_values("Data da pior queda"))

,Data da pior queda,Variação (%)
BBDC4.SA,2009-05-29 00:00:00-03:00,-43.25
B3SA3.SA,2011-05-09 00:00:00-03:00,-66.13
VALE3.SA,2019-01-28 00:00:00-02:00,-24.56
PETR3.SA,2020-03-09 00:00:00-03:00,-29.68
PETR4.SA,2020-03-09 00:00:00-03:00,-29.70
ABEV3.SA,2020-03-12 00:00:00-03:00,-15.78
ITUB4.SA,2021-10-04 00:00:00-03:00,-17.96
AXIA3.SA,2025-12-22 00:00:00-03:00,-22.53
RENT3.SA,2026-10-26 00:00:00-03:00,-39.58
BBAS3.SA,2026-10-26 00:00:00-03:00,-29.33


### 3.5 O dia seguinte a uma queda (T7)

**Perguntas:**
- Qual é a probabilidade de alta **em um dia qualquer**? Essa é a referência: sem ela, "sobe depois de cair" não diz nada.
- Qual é a probabilidade de alta **no dia seguinte a uma queda**? Como alinhar "hoje" com "amanhã" no pandas (`shift(-1)`)?
- A diferença entre as duas probabilidades é grande comparada com a variação entre os ativos? Com quantos dias de amostra?

> **Enunciado do professor (T7):** Analisando esta série temporal, você conseguira estimar avaliar se existe uma maior probabilidade maior dos ativos performarem bem logo após um dia de queda?

Código da tarefa T7

In [85]:
caiu_ontem = df_variacao.shift(1) < 0

display(pd.DataFrame({
    "Dias após queda": caiu_ontem.sum(),
    "P(alta) em um dia qualquer (%)": (df_variacao > 0).sum() / df_variacao.notna().sum() * 100,
    "P(alta) após queda (%)": (df_variacao[caiu_ontem] > 0).sum() / caiu_ontem.sum() * 100,
    "Retorno médio em um dia qualquer (%)": df_variacao.mean() * 100,
    "Retorno médio após queda (%)": df_variacao[caiu_ontem].mean() * 100,
}).round(3))

,Dias após queda,P(alta) em um dia qualquer (%),P(alta) após queda (%),Retorno médio em um dia qualquer (%),Retorno médio após queda (%)
RENT3.SA,2269,49.542,51.124,0.084,0.135
AXIA3.SA,2301,48.902,49.239,0.076,0.038
BBAS3.SA,2259,49.797,49.270,0.057,-0.002
ABEV3.SA,2210,49.840,51.493,0.056,0.116
B3SA3.SA,2268,49.221,49.868,0.043,0.132
BBDC4.SA,2201,48.412,48.932,0.036,0.071
PETR3.SA,2248,50.373,50.934,0.070,0.032
ITUB4.SA,2262,50.011,50.398,0.065,0.074
PETR4.SA,2196,50.799,51.503,0.073,0.045
VALE3.SA,2305,49.435,49.024,0.055,0.006


### 3.6 Quedas maiores que 1% (T8)

**Perguntas:**
- Qual é o **retorno médio** no pregão seguinte a quedas de mais de 1%, por ativo? E a **mediana**? (Poucos dias extremos podem puxar a média.)
- Quantas ocorrências sustentam cada média? Um ativo com 30 quedas e outro com 300 têm a mesma confiança?
- Comprar depois da queda seria uma boa estratégia? Que custos a análise ignora (corretagem, imposto, comprar exatamente no fechamento)? (Géron 1.11)
- A lista de ativos é a **carteira de 2022** aplicada a todo o passado. Empresas que saíram do índice (ou faliram) não estão na amostra: isso é **viés de sobrevivência** (Sobrevivência). Em que direção ele distorce o resultado?

> **Enunciado do professor (T8):** Seguindo a lógica da questão anterior, avalie como em média se comportam as ações no pregão seguinte a quedas maiores que 1%. De posse destes dados, você acredita que seria uma boa estratégia de investimento comprar ações que cairam no dia anterior?

Código da tarefa T8

In [86]:
caiu_mais_1 = df_variacao.shift(1) < -0.01

display(pd.DataFrame({
    "Quedas > 1%": caiu_mais_1.sum(),
    "P(alta) no dia seguinte (%)": (df_variacao[caiu_mais_1] > 0).sum() / caiu_mais_1.sum() * 100,
    "Retorno médio no dia seguinte (%)": df_variacao[caiu_mais_1].mean() * 100,
    "Mediana no dia seguinte (%)": df_variacao[caiu_mais_1].median() * 100,
    "Retorno médio em um dia qualquer (%)": df_variacao.mean() * 100,
}).round(3))

,Quedas > 1%,P(alta) no dia seguinte (%),Retorno médio no dia seguinte (%),Mediana no dia seguinte (%),Retorno médio em um dia qualquer (%)
RENT3.SA,1406,51.991,0.152,0.127,0.084
AXIA3.SA,1463,49.556,0.053,0.000,0.076
BBAS3.SA,1355,49.299,-0.010,0.000,0.057
ABEV3.SA,1006,53.479,0.189,0.154,0.056
B3SA3.SA,1432,50.279,0.156,0.049,0.043
BBDC4.SA,1223,49.877,0.140,0.000,0.036
PETR3.SA,1374,52.329,0.045,0.119,0.070
ITUB4.SA,1211,50.537,0.079,0.034,0.065
PETR4.SA,1357,52.469,0.010,0.138,0.073
VALE3.SA,1364,48.534,-0.036,0.000,0.055


### 3.7 Dividendos e *dividend yield* (T9, desafio)

**Perguntas:**
- Como baixar o histórico de dividendos de cada ativo (`get_dividends()`) e somá-los **por ano**?
- Qual preço vai no denominador do DY: fechamento do fim do ano, média do ano, primeiro dia? As convenções variam (DY: "not fully standardised"). Escolha uma e **declare**.
- Se o preço do `history()` já é ajustado (`auto_adjust=True`), ele serve para o denominador do DY, ou você precisa do preço **não ajustado**? Por quê?

> **Enunciado do professor (T9):** De tempos em tempos, empresas podem distribuir lucro aos acionistas na forma de dividendos. Podemos obter o histórico de pagamentos de dividendos usando a yfinance. Um indicador muito utilizado no mercado é o Dividend Yield (DY), que é calculado somando-se o total pago de dividendos no ano e dividindo pela cotação da ação. Calcule o DY anual de cada ativo.

Código da tarefa T9.

In [87]:
precos_sem_ajuste = {}
for ticker in tickers.tickers.keys():
    precos_sem_ajuste[ticker] = tickers.tickers[ticker].history(period="max", auto_adjust=False)

df_dividendos = pd.concat({ticker: precos_sem_ajuste[ticker]["Dividends"] for ticker in precos_sem_ajuste}, axis=1)
df_preco_real = pd.concat({ticker: precos_sem_ajuste[ticker]["Close"] for ticker in precos_sem_ajuste}, axis=1)

dividendos_ano = df_dividendos.groupby(df_dividendos.index.year).sum()
ultimo_preco_ano = df_preco_real.groupby(df_preco_real.index.year).last()

ultimo_ano_completo = pd.Timestamp.today().year - 1
df_dy = (dividendos_ano / ultimo_preco_ano * 100).loc[primeira_data.year:ultimo_ano_completo]
display(df_dy.round(2))

/tmp/ipykernel_2424937/3556441637.py:5: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  df_dividendos = pd.concat({ticker: precos_sem_ajuste[ticker]["Dividends"] for ticker in precos_sem_ajuste}, axis=1)
/tmp/ipykernel_2424937/3556441637.py:6: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  df_preco_real = pd.concat({ticker: precos_sem_ajuste[ticker]["Close"] for ticker in precos_sem_ajuste}, axis=1)


,RENT3.SA,AXIA3.SA,BBAS3.SA,ABEV3.SA,B3SA3.SA,BBDC4.SA,PETR3.SA,ITUB4.SA,PETR4.SA,VALE3.SA
Date,,,,,,,,,,
2008,5.69,1.64,1.53,5.20,0.40,1.73,1.38,0.38,1.13,3.94
2009,0.83,4.25,3.00,3.62,0.95,0.94,2.53,0.32,2.87,7.24
2010,0.76,16.53,5.69,2.01,1.19,1.12,3.19,2.30,3.64,0.76
2011,1.37,0.00,7.84,3.13,5.68,3.48,3.95,3.82,4.28,4.72
2012,1.15,0.00,6.35,3.03,3.37,2.83,2.58,3.49,2.57,4.11
2013,4.35,24.61,9.74,1.48,5.22,2.98,0.00,3.12,4.57,3.07
2014,1.35,7.17,6.98,4.26,4.11,3.09,5.61,2.81,9.95,8.63
2015,2.69,0.00,13.99,8.85,7.34,7.19,0.00,5.67,0.00,7.49
2016,2.61,0.00,3.25,3.90,3.05,4.55,0.00,4.71,0.00,0.65


### 3.8 Carteira das 3 maiores pagadoras de dividendos (T10, desafio)

**Perguntas:**
- Para montar a carteira de um ano, você usa **só o DY do ano anterior**? Se usasse o DY do próprio ano, estaria usando uma informação que ainda não existia no momento da compra (*look-ahead*). (Backtest)
- Como medir o desempenho: retorno de cada ano da carteira, retorno acumulado? Pesos iguais entre as 3 ações?
- O retorno inclui os dividendos? Se o preço já é ajustado, somar os dividendos de novo **conta duas vezes**.
- Qual é a "carteira global" de comparação: os 10 ativos com pesos iguais? Rebalanceada a cada ano?
- O resultado sobrevive ao **viés de sobrevivência** (a lista de 2022) e ao risco de *overfitting* de escolher a regra olhando o passado? (Sobrevivência; Backtest)

> **Enunciado do professor (T10):** Avalie o desempenho de uma estratégia de investimento que monta uma carteira de ações ano a ano com as 3 maiores pagadoras de dividendo do ano anterior. Compare o desempenho dela com o desempenho de uma carteira global.

Código da tarefa T10.

In [88]:
import plotly.graph_objects as go

preco_ajustado_ano = df_comum.groupby(df_comum.index.year).last()
retorno_ano = preco_ajustado_ano.pct_change()

retorno_estrategia = {}
for ano in df_dy.index:
    if ano + 1 <= ultimo_ano_completo:
        escolhidas = df_dy.loc[ano].nlargest(3).index
        retorno_estrategia[ano + 1] = retorno_ano.loc[ano + 1, escolhidas].mean()

anos = list(retorno_estrategia.keys())
df_comparacao = pd.DataFrame({
    "Estratégia DY (top 3)": pd.Series(retorno_estrategia),
    "Carteira global (10 ativos)": retorno_ano.loc[anos].mean(axis=1),
})
display((df_comparacao * 100).round(2))

display(pd.DataFrame({
    "Retorno acumulado (%)": ((1 + df_comparacao).prod() - 1) * 100,
    "Retorno médio anual (%)": df_comparacao.mean() * 100,
    "Desvio anual (%)": df_comparacao.std() * 100,
}).round(2))
display(f"Anos em que a estratégia venceu: {(df_comparacao.iloc[:, 0] > df_comparacao.iloc[:, 1]).sum()} de {len(df_comparacao)}")

acumulado = (1 + df_comparacao).cumprod()
fig = go.Figure()
for col in acumulado.columns:
    fig.add_trace(go.Scatter(x=acumulado.index, y=acumulado[col], name=col))
fig.update_layout(title="Valor de R$ 1 investido", height=450)

,Estratégia DY (top 3),Carteira global (10 ativos)
2009,112.40,77.30
2010,9.41,4.63
2011,-19.01,-16.54
2012,25.50,11.88
2013,0.59,-2.56
2014,0.27,-5.53
2015,-23.74,-14.50
2016,64.29,93.02
2017,32.48,31.79
2018,31.43,30.43


,Retorno acumulado (%),Retorno médio anual (%),Desvio anual (%)
Estratégia DY (top 3),1182.68,21.07,37.37
Carteira global (10 ativos),1080.12,19.04,30.97


'Anos em que a estratégia venceu: 10 de 17'

## 7. Organizar apresentação da solução

> TODO (Gabriel).

**Perguntas** (Géron, etapa 7):
- Qual é a resposta curta para cada pergunta do README?
- Quais são as **limitações** que mudam a interpretação: carteira de 2022 aplicada ao passado, preço ajustado, custos ignorados, tamanho da amostra?
- O que você aprendeu (Géron 3.10)?